[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaphaelRAY/covid-brazil-analysis/blob/main/01_primeira_carga.ipynb)

# Carga dados covid Big Query (BQ)

## Carregar módulos

In [1]:
from google.colab import auth
from google.cloud import bigquery

In [2]:
import numpy as np
import pandas as pd

## Autenticar projeto

In [3]:
auth.authenticate_user()

In [4]:
project_id = 'projetotestemaua5584'

In [5]:
client = bigquery.Client(project=project_id)

## Carregar dados

- Dados Covid Brasil

In [6]:
! wget --no-check-certificate --content-disposition 'https://github.com/wcota/covid19br/blob/master/cases-brazil-cities-time.csv.gz?raw=true'

--2025-05-19 22:13:34--  https://github.com/wcota/covid19br/blob/master/cases-brazil-cities-time.csv.gz?raw=true
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/wcota/covid19br/raw/refs/heads/master/cases-brazil-cities-time.csv.gz [following]
--2025-05-19 22:13:34--  https://github.com/wcota/covid19br/raw/refs/heads/master/cases-brazil-cities-time.csv.gz
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/wcota/covid19br/refs/heads/master/cases-brazil-cities-time.csv.gz [following]
--2025-05-19 22:13:34--  https://raw.githubusercontent.com/wcota/covid19br/refs/heads/master/cases-brazil-cities-time.csv.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubus

In [7]:
! gunzip cases-brazil-cities-time.csv.gz

In [8]:
dados_brasil = pd.read_csv('cases-brazil-cities-time.csv')

- Dados censo

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
import os
if not os.path.exists('dados_municipios_2010.csv'):
    print('Please upload dados_municipios_2010.csv to the Colab environment.')


In [ ]:
censo_path = 'dados_municipios_2010.csv'
dados_censo = pd.read_csv(censo_path, sep = ';', decimal = ',', encoding = 'latin1')

## Ajustar dados

In [ ]:
dados_brasil = dados_brasil[dados_brasil['state'] != 'TOTAL']

In [ ]:
cols = ['ibgeID', 'date', 'state', 'city', 'totalCases', 'deaths']

In [ ]:
dados_brasil = dados_brasil[cols].reset_index(drop=True)

## Adicionar dados covid no BQ

In [ ]:
# # se criou anteriormente no console
# dataset_ref = client.dataset('dados_brasil')

In [ ]:
# criar no python
try:
    dataset_ref = client.dataset('dados_brasil')
    client.get_dataset(dataset_ref)
    print("Dataset already exists")
except Exception as e:
    dataset_ref = bigquery.Dataset(project_id+'.dados_brasil')
    dataset_ref = client.create_dataset(dataset_ref)
    print("Dataset created")

In [ ]:
table_ref = dataset_ref.table("dados_brasil_covid")

In [ ]:
job = client.load_table_from_dataframe(dados_brasil, table_ref,
                                       job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()

/usr/local/lib/python3.7/dist-packages/google/cloud/bigquery/_pandas_helpers.py:275: UserWarning: Unable to determine type of column 'date'.
  warnings.warn(u"Unable to determine type of column '{}'.".format(column))


## Adicionar dados do censo no BQ

In [ ]:
table_ref = dataset_ref.table("dados_brasil_censo")

In [ ]:
job = client.load_table_from_dataframe(dados_censo, table_ref,
                                       job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()

/usr/local/lib/python3.7/dist-packages/google/cloud/bigquery/_pandas_helpers.py:275: UserWarning: Unable to determine type of column 'Municipio'.
  warnings.warn(u"Unable to determine type of column '{}'.".format(column))


In [ ]:
# # usar quando gerar processo de atualizacao
# job = client.load_table_from_dataframe(players_view, table_ref)
# job.result()